# Stage 4 — Evidence fusion and risk estimation (Stage 2 + 2B + Stage 3 only)

**GPU: not needed.** Everything here is a small scikit-learn model on about 20 numbers per image. Use a CPU runtime.

This follows your architecture, trimmed as you asked: **no visual-anomaly branch**. Evidence comes from
- **Stage 2** (OCR text risk, similarity scores, OCR quality): `dev_stage2.jsonl`
- **Stage 2B** (hidden-text probability, rescue-OCR risk), optional: `dev_stage2b.jsonl` + `hidden_feat.csv`
- **Stage 3** (cross-view consistency): `dev_stage3.jsonl`

An **MLP** fuses them into one **risk score** (0-1). Bands from your diagram: **Low < 0.3 -> Allow**, **Medium 0.3-0.7 -> Sanitize**, **High >= 0.7 -> Quarantine**.

## Paths (same Drive folder as Stage 2 / Stage 3)
| File | Path |
|---|---|
| labels | `/content/drive/MyDrive/ipi_dataset_out/meta.csv` |
| Stage 2 | `.../ipi_dataset_out/stage2_out/dev_stage2.jsonl` (backup: `.../ipi_dataset_out/dev_stage2.jsonl`) |
| Stage 2B | `.../stage2_out/stage2b/dev_stage2b.jsonl` and `.../stage2b/hidden_feat.csv` |
| Stage 3 | `.../ipi_dataset_out/dev_stage3.jsonl` (written by the Stage 3 notebook) |
| Stage 4 output | `.../ipi_dataset_out/stage4_out/fusion_model.joblib` |

## Rules that keep the numbers honest
1. **Nothing from the CSV labels is used as an input feature.**
2. **Every score is out-of-fold**, with folds grouped by attack string and by source photo, so the model never sees the same string or photo in training and scoring.
3. **`hidden_p` is rebuilt out-of-fold.** Your Stage 2B detector was trained and scored on the same dev images, so its own `hidden_p` is over-confident on dev; feeding it to Stage 4 would leak.
4. **An ablation table + bootstrap interval** shows whether each evidence source really helps. Keep a source only if its interval stays above 0.
5. Test is touched once, at the end, with the frozen model.

## 1. Setup and paths

In [1]:
# ---- Stage 4 Cell 1: setup (CPU only, no models to load) ----
import os, json, shutil, warnings, joblib
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, roc_curve
warnings.filterwarnings('ignore')

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

DATA_ROOT = Path('/content/drive/MyDrive/ipi_dataset_out')
S2_ROOT   = DATA_ROOT / 'stage2_out'
S2B_ROOT  = S2_ROOT / 'stage2b'
S3_ROOT   = DATA_ROOT / 'stage3_out'
S4_ROOT   = DATA_ROOT / 'stage4_out'; S4_ROOT.mkdir(exist_ok=True)

def first_existing(*paths):
    for p in paths:
        if Path(p).exists(): return Path(p)
    return None

P = {'meta':        first_existing(DATA_ROOT / 'meta.csv'),
     'dev_stage2':  first_existing(S2_ROOT / 'dev_stage2.jsonl', DATA_ROOT / 'dev_stage2.jsonl'),
     'dev_stage3':  first_existing(S3_ROOT / 'dev_stage3.jsonl', DATA_ROOT / 'dev_stage3.jsonl'),
     'dev_stage2b': first_existing(S2B_ROOT / 'dev_stage2b.jsonl', DATA_ROOT / 'dev_stage2b.jsonl'),   # optional
     'hidden_feat': first_existing(S2B_ROOT / 'hidden_feat.csv', DATA_ROOT / 'hidden_feat.csv')}        # optional (for honest hidden_p)
for k, v in P.items(): print(f'{k:12s}', 'OK     ' if v else 'MISSING', v or '')
assert P['meta'] and P['dev_stage2'] and P['dev_stage3'], 'meta.csv, dev_stage2.jsonl and dev_stage3.jsonl are required (run Stage 2 and Stage 3 first)'


Mounted at /content/drive
meta         OK      /content/drive/MyDrive/ipi_dataset_out/meta.csv
dev_stage2   OK      /content/drive/MyDrive/ipi_dataset_out/stage2_out/dev_stage2.jsonl
dev_stage3   OK      /content/drive/MyDrive/ipi_dataset_out/stage3_out/dev_stage3.jsonl
dev_stage2b  OK      /content/drive/MyDrive/ipi_dataset_out/stage2_out/stage2b/dev_stage2b.jsonl
hidden_feat  OK      /content/drive/MyDrive/ipi_dataset_out/stage2_out/stage2b/hidden_feat.csv


## 2. Loaders

In [2]:
# ---- Stage 4 Cell 2: loaders (lean: the big raw-OCR fields are not kept) ----
S2_FEATS  = ['text_risk', 'semantic_risk', 'sim_emb', 'sim_char', 'sim_query', 'n_text_regions', 'n_unreadable', 'mean_conf', 'n_chars', 'no_text']
S2B_FEATS = ['hidden_p', 's2b_semantic_risk', 'risk_max_2b']
S3_FEATS  = ['xv_presence', 'xv_text', 'xv_revealed', 'xv_score', 'xv_inconsistent_risk', 'n_regions', 'n_inconsistent']

def load_labels(path):
    L = pd.read_csv(path, sep=None, engine='python'); L['key'] = L['file'].map(lambda f: Path(str(f)).name); return L

def load_s2(path):
    keep = ['file', 'text_risk', 'semantic_risk', 'sim_emb', 'sim_char', 'sim_query', 'n_text_regions', 'n_unreadable', 'no_text']
    rows = []
    for l in open(path):
        d = json.loads(l); r = {k: d.get(k) for k in keep}
        tb = d.get('text_boxes') or []
        r['mean_conf'] = float(np.mean([b['conf'] for b in tb])) if tb else 0.0
        r['n_chars'] = len(d.get('extracted_text') or '')
        rows.append(r)
    F = pd.DataFrame(rows); F['no_text'] = F['no_text'].astype(int); return F

def load_cols(path, cols):
    return pd.DataFrame([{c: json.loads(l).get(c) for c in cols} for l in open(path)])

def oof_hidden_p(hidden_feat_csv, labels, seed=0):
    """Honest hidden-text probability for every dev image: each image is scored by a detector that never saw its photo.
    (The Stage 2B detector in your run was trained AND scored on dev, so its own hidden_p is over-confident on dev.)"""
    from sklearn.ensemble import HistGradientBoostingClassifier as HGB
    feat = pd.read_csv(hidden_feat_csv)
    m = feat.merge(labels[['key', 'mode', 'clean_file']], left_on='file', right_on='key')
    y = m['mode'].isin(('faint', 'watermark')).astype(int).values
    cols = [c for c in feat.columns if c != 'file']
    groups = m['clean_file'].astype(str).where(m['clean_file'].notna(), m['file'])
    oof = np.zeros(len(m))
    for a, b in GroupKFold(5).split(m, y, groups):
        oof[b] = HGB(max_depth=4, learning_rate=0.05, max_iter=300, class_weight='balanced', random_state=seed) \
                    .fit(m.iloc[a][cols], y[a]).predict_proba(m.iloc[b][cols])[:, 1]
    print(f'hidden_p (out-of-fold) built for {len(m)} images; AUROC hidden-vs-rest = {roc_auc_score(y, oof):.3f}')
    return pd.Series(oof, index=m['file'].values)

def build_table(s2, s3, s2b, hid, labels):
    F = s2.merge(s3[['file'] + [c for c in S3_FEATS if c in s3]], on='file', how='left')
    for c in S3_FEATS: F[c] = F[c].fillna(0.0) if c in F else 0.0
    F = F.merge(s2b[['file', 's2b_semantic_risk']], on='file', how='left') if s2b is not None else F.assign(s2b_semantic_risk=0.0)
    F['s2b_semantic_risk'] = F['s2b_semantic_risk'].fillna(0.0)
    F['hidden_p'] = F['file'].map(hid).fillna(0.0) if hid is not None else 0.0
    F['risk_max_2b'] = F[['semantic_risk', 's2b_semantic_risk']].max(axis=1)         # Stage 2 / 2B late fusion as one feature
    F = F.merge(labels[['key', 'kind', 'mode', 'category', 'text_id', 'clean_file']], left_on='file', right_on='key', how='left')
    F['y'] = (F['kind'] == 'attack').astype(int)
    g = ('t:' + F['text_id'].astype(str)).where(F['text_id'].notna(), 'c:' + F['clean_file'].astype(str))
    F['group'] = g.where(F['text_id'].notna() | F['clean_file'].notna(), 'f:' + F['file'])      # same string / same photo never in train AND test fold
    return F


## 3. Models and metrics

In [3]:
# ---- Stage 4 Cell 3: fusion models, cross-validation, metrics ----
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

LOW_T, HIGH_T = 0.3, 0.7                      # risk bands from your architecture: Low 0-0.3, Medium 0.3-0.7, High 0.7-1.0
ACTION = {'Low': 'Allow', 'Medium': 'Sanitize', 'High': 'Quarantine'}

def make_model(kind, seed=0):
    if kind == 'mlp':
        return make_pipeline(StandardScaler(), MLPClassifier((32, 16), alpha=1e-2, max_iter=600, early_stopping=True,
                                                             n_iter_no_change=20, random_state=seed))
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))

def cv_scores(F, feats, kind='mlp', seeds=(0, 1, 2), n_splits=5):
    """Out-of-fold risk score for every image (each image is scored by models that never saw its string or photo)."""
    oof = np.zeros(len(F)); seeds = seeds if kind == 'mlp' else (0,)
    for tr, te in GroupKFold(n_splits).split(F, F['y'], F['group']):
        ps = [make_model(kind, s).fit(F.iloc[tr][feats], F['y'].iloc[tr]).predict_proba(F.iloc[te][feats])[:, 1] for s in seeds]
        oof[te] = np.mean(ps, axis=0)
    return oof

def tpr_at_fpr(y, p, target=0.05):
    fpr, tpr, th = roc_curve(y, p); k = np.searchsorted(fpr, target, side='right') - 1
    return float(tpr[k]), float(th[k])

def metrics_row(F, p, name):
    y = F['y'].values; tpr, thr = tpr_at_fpr(y, p); flag = p >= thr
    row = {'set': name, 'AUROC': round(roc_auc_score(y, p), 3), 'TPR@5%FPR': round(tpr, 3)}
    for m in ('faint', 'watermark', 'rotated'):
        s = ((F['mode'] == m) & (F['y'] == 1)).values; row[f'{m}_caught'] = round(float(flag[s].mean()), 3) if s.any() else np.nan
    return row

def boot_auc_diff(F, p_a, p_b, B=300, seed=0):
    """95% interval of AUROC(b) - AUROC(a), resampling whole groups (strings/photos) so the interval is not too narrow."""
    rng = np.random.default_rng(seed); g = F['group'].values; ug = np.unique(g)
    idx = {k: np.where(g == k)[0] for k in ug}; y = F['y'].values; d = []
    for _ in range(B):
        ii = np.concatenate([idx[k] for k in rng.choice(ug, len(ug))])
        if y[ii].min() == y[ii].max(): continue
        d.append(roc_auc_score(y[ii], p_b[ii]) - roc_auc_score(y[ii], p_a[ii]))
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def band(p): return np.where(p < LOW_T, 'Low', np.where(p < HIGH_T, 'Medium', 'High'))

def policy_report(F, p):
    T = F.assign(band=band(p)); T['action'] = T['band'].map(ACTION)
    cols = ['Low', 'Medium', 'High']
    print('Share of each image kind that lands in each risk band  (rows sum to 1):')
    print(pd.crosstab(T['kind'], T['band'], normalize='index').reindex(columns=cols, fill_value=0).round(2).to_string())
    print('\nAttacks by mode: share Allowed (Low = missed) / Sanitized / Quarantined:')
    a = T[T['y'] == 1]
    print(pd.crosstab(a['mode'], a['band'], normalize='index').reindex(columns=cols, fill_value=0).round(2).to_string())
    print('\nBenign + clean images that would be Quarantined (over-blocking):',
          f"{(T[T['y'] == 0]['band'] == 'High').mean():.3f}", '| Sanitized:', f"{(T[T['y'] == 0]['band'] == 'Medium').mean():.3f}")
    return T


## 4. Build the evidence table

In [4]:
# ---- Stage 4 Cell 4: load Stage 2 / 2B / 3 and build the evidence table ----
labels = load_labels(P['meta'])
s2  = load_s2(P['dev_stage2'])
s3  = pd.read_json(P['dev_stage3'], lines=True)
s2b = load_cols(P['dev_stage2b'], ['file', 'hidden_p', 's2b_semantic_risk']) if P['dev_stage2b'] else None
hid = oof_hidden_p(P['hidden_feat'], labels) if P['hidden_feat'] else None
USE_2B = (s2b is not None) and (hid is not None)
if not USE_2B: print('Stage 2B inputs missing -> Stage 4 will run on Stage 2 + Stage 3 only')

F = build_table(s2, s3, s2b, hid, labels)
print(len(F), 'images | attacks', int(F['y'].sum()), '| benign_text', int((F['kind'] == 'benign_text').sum()), '| clean', int((F['kind'] == 'clean').sum()))
print('missing Stage 3 rows:', int(F['xv_score'].isna().sum()), '| images with label NaN:', int(F['kind'].isna().sum()))
print('(attack share in this sample is set by your Stage 2 sampling, not by real-world prevalence: read the rates below, not "precision")')

FEATS = {'S2': S2_FEATS, 'S2+S3': S2_FEATS + S3_FEATS}
if USE_2B:
    FEATS.update({'S2+2B': S2_FEATS + S2B_FEATS, 'S2+2B+S3': S2_FEATS + S2B_FEATS + S3_FEATS})
for k in FEATS: FEATS[k] = [c for c in FEATS[k] if c in F]


hidden_p (out-of-fold) built for 2997 images; AUROC hidden-vs-rest = 0.577
1106 images | attacks 600 | benign_text 406 | clean 100
missing Stage 3 rows: 0 | images with label NaN: 0
(attack share in this sample is set by your Stage 2 sampling, not by real-world prevalence: read the rates below, not "precision")


## 5. Ablation: Stage 2 alone vs +2B vs +Stage 3 vs everything
`TPR@5%FPR` = share of attacks caught when only 5% of harmless images are flagged; it makes feature sets comparable. The `faint` / `watermark` columns are the blind spots you are targeting.

In [5]:
# ---- Stage 4 Cell 5: does each evidence source actually help?  (all numbers are out-of-fold, grouped by string/photo) ----
MODEL = 'mlp'                     # architecture uses an MLP; 'logreg' is the sanity check printed below
oof = {}
rows = [metrics_row(F, F['semantic_risk'].values, 'baseline: Stage 2 semantic_risk alone (no learning)')]
for name, feats in FEATS.items():
    for kind in ('logreg', MODEL):
        oof[(name, kind)] = cv_scores(F, feats, kind)
        rows.append(metrics_row(F, oof[(name, kind)], f'{name} [{kind}]'))
print(pd.DataFrame(rows).to_string(index=False))

base = oof[('S2', MODEL)]
print('\nAUROC change vs the Stage-2-only model (mean, 95% interval; if the interval includes 0 the gain is not proven):')
for name in FEATS:
    if name == 'S2': continue
    d, lo, hi = boot_auc_diff(F, base, oof[(name, MODEL)]); print(f'  {name:10s} {d:+.3f}   [{lo:+.3f}, {hi:+.3f}]')


                                                set  AUROC  TPR@5%FPR  faint_caught  watermark_caught  rotated_caught
baseline: Stage 2 semantic_risk alone (no learning)  0.914      0.790          0.71              0.54            0.71
                                        S2 [logreg]  0.947      0.797          0.68              0.56            0.74
                                           S2 [mlp]  0.931      0.783          0.61              0.55            0.74
                                     S2+S3 [logreg]  0.949      0.788          0.69              0.54            0.75
                                        S2+S3 [mlp]  0.942      0.785          0.67              0.55            0.71
                                     S2+2B [logreg]  0.946      0.793          0.67              0.58            0.74
                                        S2+2B [mlp]  0.943      0.788          0.70              0.59            0.70
                                  S2+2B+S3 [logreg]  0.9

## 6. Risk bands and policy
Change `FINAL_SET` to the smallest set the ablation supports.

In [6]:
# ---- Stage 4 Cell 6: risk score -> Low / Medium / High -> Allow / Sanitize / Quarantine ----
FINAL_SET = 'S2+2B+S3' if USE_2B else 'S2+S3'      # change after reading the ablation above (use the smallest set that is clearly best)
p_final = oof[(FINAL_SET, MODEL)]
print('Risk bands:  Low < %.1f -> Allow | Medium %.1f-%.1f -> Sanitize | High >= %.1f -> Quarantine   (feature set: %s)\n' % (LOW_T, LOW_T, HIGH_T, HIGH_T, FINAL_SET))
T = policy_report(F, p_final)


Risk bands:  Low < 0.3 -> Allow | Medium 0.3-0.7 -> Sanitize | High >= 0.7 -> Quarantine   (feature set: S2+2B+S3)

Share of each image kind that lands in each risk band  (rows sum to 1):
band          Low  Medium  High
kind                           
attack       0.06    0.22  0.72
benign_text  0.67    0.29  0.03
clean        0.95    0.05  0.00

Attacks by mode: share Allowed (Low = missed) / Sanitized / Quarantined:
band            Low  Medium  High
mode                             
banner         0.03    0.07  0.90
faint          0.10    0.27  0.63
rotated        0.07    0.33  0.60
small_corner   0.01    0.20  0.79
visible_large  0.00    0.10  0.90
watermark      0.17    0.34  0.49

Benign + clean images that would be Quarantined (over-blocking): 0.026 | Sanitized: 0.245


## 6b. Risk estimation for each image
Shows every image's **risk score, Low / Medium / High level and action**, plus `assess_image(name)` to see the evidence (Stage 2, 2B, 3) behind one image's score.

In [7]:
# ---- Stage 4 Cell 6b: risk estimation PER IMAGE  (risk score -> Low / Medium / High -> Allow / Sanitize / Quarantine) ----
R = F[['file', 'kind', 'mode']].copy()
R['risk_score'] = np.round(p_final, 3)
R['risk_level'] = band(p_final)
R['action']     = R['risk_level'].map(ACTION)
pd.set_option('display.width', 170); pd.set_option('display.max_colwidth', 40)

print('How many images in each risk level:')
print(R['risk_level'].value_counts().reindex(['Low', 'Medium', 'High']).to_string())
print('\nHIGH risk examples (highest scores):');   print(R.sort_values('risk_score', ascending=False).head(6).to_string(index=False))
print('\nMEDIUM risk examples:');                  print(R[R['risk_level'] == 'Medium'].sample(6, random_state=0).to_string(index=False))
print('\nLOW risk but actually an attack (missed by fusion):'); print(R[(R['risk_level'] == 'Low') & (R['kind'] == 'attack')].head(6).to_string(index=False))

def assess_image(name):
    """Show the evidence behind one image's risk estimate."""
    i = F.index[F['file'] == name][0]; r = F.loc[i]; s = float(p_final[i]); lv = band(np.array([s]))[0]
    print(f'{name}\n  RISK {s:.2f}  ->  {lv}  ->  {ACTION[lv]}')
    print(f"  Stage 2 : semantic_risk {r['semantic_risk']:.2f} | text_risk {r['text_risk']:.2f} | OCR confidence {r['mean_conf']:.2f} | unreadable regions {int(r['n_unreadable'])}")
    print(f"  Stage 2B: hidden_text_prob {r['hidden_p']:.2f} | rescue-OCR risk {r['s2b_semantic_risk']:.2f}")
    print(f"  Stage 3 : cross-view score {r['xv_score']:.2f} | text disagreement {r['xv_text']:.2f} | disputed-region risk {r['xv_inconsistent_risk']:.2f}")
    print(f"  (dev label, for checking only) kind={r['kind']}  mode={r['mode']}")

assess_image(R.sort_values('risk_score', ascending=False)['file'].iloc[0])      # try any file name: assess_image('xxxx.png')


How many images in each risk level:
risk_level
Low       407
Medium    255
High      444

HIGH risk examples (highest scores):
                              file        kind          mode  risk_score risk_level     action
       000000236599_banner_atk.png      attack        banner       1.000       High Quarantine
 000000478474_small_corner_atk.png      attack  small_corner       1.000       High Quarantine
    000000310980_faint_d40_atk.png      attack         faint       1.000       High Quarantine
    000000578922_faint_d24_atk.png      attack         faint       0.999       High Quarantine
    000000314914_watermark_ben.png benign_text     watermark       0.998       High Quarantine
000000459887_visible_large_atk.png      attack visible_large       0.997       High Quarantine

MEDIUM risk examples:
                             file        kind         mode  risk_score risk_level   action
   000000004765_faint_d40_ben.png benign_text        faint       0.375     Medium Sanitize
000

## 7. Fit the final model on all of dev and save it

In [8]:
# ---- Stage 4 Cell 7: fit the final fusion model on all of dev and save it ----
feats = FEATS[FINAL_SET]
models = [make_model(MODEL, s).fit(F[feats], F['y']) for s in (0, 1, 2, 3, 4)]
def risk_score(X, models=models, feats=feats): return np.mean([m.predict_proba(X[feats])[:, 1] for m in models], axis=0)
joblib.dump({'models': models, 'feats': feats, 'set': FINAL_SET, 'kind': MODEL, 'bands': (LOW_T, HIGH_T)}, S4_ROOT / 'fusion_model.joblib')
T[['file', 'kind', 'mode', 'y', 'band', 'action']].assign(risk=p_final).to_csv(S4_ROOT / 'dev_stage4_oof.csv', index=False)
print('saved model ->', S4_ROOT / 'fusion_model.joblib', '| out-of-fold dev scores ->', S4_ROOT / 'dev_stage4_oof.csv')


saved model -> /content/drive/MyDrive/ipi_dataset_out/stage4_out/fusion_model.joblib | out-of-fold dev scores -> /content/drive/MyDrive/ipi_dataset_out/stage4_out/dev_stage4_oof.csv


# Final test (once, after Stage 2, 2B and 3 have been run on the test split)

In [9]:
# ---- Stage 4 Cell 8: FINAL TEST (run once, after Stage 2, 2B and 3 have been run on the test split) ----
RUN_TEST = False
if RUN_TEST:
    bundle = joblib.load(S4_ROOT / 'fusion_model.joblib'); feats = bundle['feats']
    t2  = first_existing(S2_ROOT / 'test_stage2.jsonl', DATA_ROOT / 'test_stage2.jsonl')
    t3  = first_existing(S3_ROOT / 'test_stage3.jsonl', DATA_ROOT / 'test_stage3.jsonl')
    t2b = first_existing(S2B_ROOT / 'test_stage2b.jsonl', DATA_ROOT / 'test_stage2b.jsonl')
    s2t, s3t = load_s2(t2), pd.read_json(t3, lines=True)
    s2bt = load_cols(t2b, ['file', 'hidden_p', 's2b_semantic_risk']) if t2b else None
    hid_t = pd.Series(s2bt['hidden_p'].values, index=s2bt['file'].values) if s2bt is not None else None   # frozen Stage 2B detector, no refit
    Ft = build_table(s2t, s3t, s2bt, hid_t, labels)
    pt = np.mean([m.predict_proba(Ft[feats])[:, 1] for m in bundle['models']], axis=0)
    print('TEST AUROC:', round(roc_auc_score(Ft['y'], pt), 3), '| baseline Stage 2 semantic_risk AUROC:', round(roc_auc_score(Ft['y'], Ft['semantic_risk']), 3))
    print(metrics_row(Ft, pt, 'fusion')); print(metrics_row(Ft, Ft['semantic_risk'].values, 'baseline'))
    _ = policy_report(Ft, pt)
